# M9.1 · Dijital görüntü: örnekleme, interpolasyon, geometri

Bir görüntü, sürekli bir sahnenin örneklenmiş ve kuantalanmış hâlidir. Bit derinliğini, komşuluk ve mesafe ölçülerini, en yakın komşu ve bilineer interpolasyonu elle hesaplıyor; saf Python'da `cv2.resize`'ın piksel merkezi kuralını birebir yeniden üretiyor, NumPy'da bilineer ve bikübik yeniden boyutlandırmayı döngüsüz yazıp OpenCV ile karşılaştırıyor, PyTorch'ta `grid_sample` ile türevlenebilir afin dönüşüm yazıp iki görüntüyü gradient descent ile hizalıyor (image registration) ve en sonda yüzlerce farklı dönüşümü tek çağrıda uyguluyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-1-digital-image.html](https://m-gonul.github.io/pytorch-lab/m9-1-digital-image.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_1_digital_image.py
CHECKS = {
 "b": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"cv2/numpy kullanılmıyor\", lambda: no_imports(src, {\"cv2\", \"numpy\", \"scipy\", \"skimage\"}))\n    R.test(\"bilinear_at (elle örnek)\", lambda: close(need(ns, \"bilinear_at\")([[10.0, 20.0], [30.0, 40.0]], 0.25, 0.5), 22.5))\n    R.test(\"bilinear_at: kenar kırpma\", lambda: close(need(ns, \"bilinear_at\")([[10.0, 20.0], [30.0, 40.0]], 5.0, -3.0), 20.0))\n    crop = data.camera()[200:224, 240:272].astype(np.float32)\n    L = crop.tolist()\n\n    def nn():\n        for H2, W2 in ((12, 16), (40, 50)):\n            got = np.array(need(ns, \"resize_nearest\")(L, H2, W2), np.float32)\n            close(got, cv2.resize(crop, (W2, H2), interpolation=cv2.INTER_NEAREST), what=f\"{H2}×{W2}\")\n    R.test(\"resize_nearest = cv2.INTER_NEAREST\", nn)\n\n    def bil():\n        for H2, W2 in ((10, 14), (48, 60), (24, 32)):\n            got = np.array(need(ns, \"resize_bilinear\")(L, H2, W2), np.float32)\n            close(got, cv2.resize(crop, (W2, H2), interpolation=cv2.INTER_LINEAR), atol=1e-3, what=f\"{H2}×{W2}\")\n    R.test(\"resize_bilinear = cv2.INTER_LINEAR (küçültme, büyütme, aynı boyut)\", bil)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time, uses_any\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"cv2.resize kullanılmıyor\", lambda: ensure(not uses_any(src, [\"resize\"]) or \"def resize\" in src and \"cv2\" not in src, \"cv2 kullanma\"))\n    img = data.camera().astype(np.float32)\n    for H2, W2 in ((200, 300), (700, 900), (512, 512)):\n        R.test(f\"bilineer {H2}×{W2} = cv2\", (lambda H2=H2, W2=W2: close(need(ns, \"resize_bilinear\")(img, H2, W2), cv2.resize(img, (W2, H2), interpolation=cv2.INTER_LINEAR), atol=1e-2)))\n    for H2, W2 in ((200, 300), (700, 900)):\n        R.test(f\"bikübik {H2}×{W2} = cv2\", (lambda H2=H2, W2=W2: close(need(ns, \"resize_cubic\")(img, H2, W2), cv2.resize(img, (W2, H2), interpolation=cv2.INTER_CUBIC), atol=1e-2)))\n    if \"resize_cubic\" in ns:\n        small = cv2.resize(img, (128, 128), interpolation=cv2.INTER_AREA)\n        rows = []\n        for name, fn in ((\"en yakın\", lambda: cv2.resize(small, (512, 512), interpolation=cv2.INTER_NEAREST)),\n                         (\"bilineer\", lambda: ns[\"resize_bilinear\"](small, 512, 512)), (\"bikübik\", lambda: ns[\"resize_cubic\"](small, 512, 512))):\n            up = fn()\n            psnr = 10 * np.log10(255 ** 2 / np.mean((up - img) ** 2))\n            rows.append([name, f\"{psnr:.2f} dB\"])\n        R.table(\"128 → 512 büyütme, orijinale göre PSNR\", [\"yöntem\", \"PSNR\"], rows,\n                note=\"Bikübik kenarları daha keskin korur; en yakın komşu blokludur.\")\n",
 "d": "import math\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    img = torch.tensor(data.camera(), dtype=torch.float64)[::4, ::4] / 255\n    H, W = img.shape\n\n    def vs_scipy():\n        th = math.radians(17)\n        A = torch.tensor([[math.cos(th), -math.sin(th)], [math.sin(th), math.cos(th)]], dtype=torch.float64) * 0.9\n        t = torch.tensor([12.0, -7.5], dtype=torch.float64)\n        got = need(ns, \"warp_affine\")(img, A, t).numpy()\n        M = np.array([[A[1, 1], A[1, 0]], [A[0, 1], A[0, 0]]])\n        ref = ndimage.affine_transform(img.numpy(), M, offset=[t[1].item(), t[0].item()], order=1, mode=\"constant\", cval=0.0)\n        inner = np.zeros_like(ref, bool); inner[8:-8, 8:-8] = True\n        S = np.stack(np.meshgrid(np.arange(W), np.arange(H)), -1) @ A.numpy().T + t.numpy()\n        ok = inner & (S[..., 0] > 1) & (S[..., 0] < W - 2) & (S[..., 1] > 1) & (S[..., 1] < H - 2)\n        close(got[ok], ref[ok], atol=1e-9, what=\"iç pikseller\")\n    R.test(\"warp_affine = scipy.ndimage.affine_transform (iç bölge)\", vs_scipy)\n\n    def ident():\n        close(need(ns, \"warp_affine\")(img, torch.eye(2, dtype=torch.float64), torch.zeros(2, dtype=torch.float64)), img, atol=1e-9)\n        sh = ns[\"warp_affine\"](img, torch.eye(2, dtype=torch.float64), torch.tensor([5.0, 0.0], dtype=torch.float64))\n        close(sh[:, :-5], img[:, 5:], atol=1e-9, what=\"5 piksel sola kaydırma\")\n    R.test(\"birim dönüşüm ve tam sayı kaydırma\", ident)\n\n    def grad():\n        A = torch.eye(2, dtype=torch.float64).requires_grad_(True)\n        t = torch.tensor([0.3, 0.2], dtype=torch.float64, requires_grad=True)\n        need(ns, \"warp_affine\")(img, A, t).sum().backward()\n        ensure(A.grad is not None and t.grad is not None and t.grad.abs().sum() > 0, \"A ve t gradyan almalı\")\n    R.test(\"türevlenebilir\", grad)\n\n    def reg():\n        fixed = img.float()\n        th, tx, ty = math.radians(8), 4.0, -3.0\n        c, s = math.cos(th), math.sin(th)\n        A = torch.tensor([[c, -s], [s, c]])\n        ctr = torch.tensor([(W - 1) / 2, (H - 1) / 2])\n        tt = ctr - A @ ctr + torch.tensor([tx, ty])\n        Ai = torch.linalg.inv(A)\n        moving = need(ns, \"warp_affine\")(fixed, Ai, -Ai @ tt)\n        f = ndimage.gaussian_filter(fixed.numpy(), 2); m = ndimage.gaussian_filter(moving.numpy(), 2)\n        est = need(ns, \"register\")(torch.tensor(f), torch.tensor(m))\n        ensure(abs(math.degrees(est[0]) - 8) < 0.5 and abs(est[1] - tx) < 0.5 and abs(est[2] - ty) < 0.5,\n               f\"bulunan θ = {math.degrees(est[0]):.2f}°, t = ({est[1]:.2f}, {est[2]:.2f}); gerçek 8°, (4, −3)\")\n        return f\"θ = {math.degrees(est[0]):.2f}° (8°), t = ({est[1]:.2f}, {est[2]:.2f}) ((4, −3))\"\n    R.test(\"register: döndürme + kaydırmayı gradient descent ile bul\", reg)\n",
 "e": "import math\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef ref_one(img, A, t):\n    H, W = img.shape\n    ys, xs = torch.meshgrid(torch.arange(H, dtype=img.dtype), torch.arange(W, dtype=img.dtype), indexing=\"ij\")\n    S = torch.stack([xs, ys], -1) @ A.T + t\n    grid = torch.stack([(2 * S[..., 0] + 1) / W - 1, (2 * S[..., 1] + 1) / H - 1], -1)[None]\n    return F.grid_sample(img[None, None], grid, align_corners=False)[0, 0]\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: max_loops(src, 0, func=\"warp_batch\"))\n    g = torch.Generator().manual_seed(0)\n    B = 16\n    imgs = torch.rand(B, 40, 56, generator=g, dtype=torch.float64)\n    th = torch.rand(B, generator=g, dtype=torch.float64) * 2 * math.pi\n    sc = 0.7 + 0.6 * torch.rand(B, generator=g, dtype=torch.float64)\n    A = torch.stack([torch.stack([th.cos(), -th.sin()], -1), torch.stack([th.sin(), th.cos()], -1)], 1) * sc[:, None, None]\n    t = torch.randn(B, 2, generator=g, dtype=torch.float64) * 5\n\n    def eq():\n        out = need(ns, \"warp_batch\")(imgs, A, t)\n        for i in range(B):\n            close(out[i], ref_one(imgs[i], A[i], t[i]), atol=1e-9, what=f\"görüntü {i}\")\n    R.test(\"her görüntü kendi dönüşümüyle\", eq)\n    if \"warp_batch\" not in ns:\n        return\n    Bb = 256\n    big = torch.rand(Bb, 64, 64, generator=g)\n    Ab, tb = A[:1].float().repeat(Bb, 1, 1), t[:1].float().repeat(Bb, 1)\n    t_loop = bench(lambda: [ref_one(big[i], Ab[i], tb[i]) for i in range(Bb)], repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"warp_batch\"](big, Ab, tb), repeat=3)\n    R.table(f\"{Bb} görüntü, 64 × 64\", [\"yöntem\", \"süre\"], [[\"görüntü başına grid_sample\", fmt_time(t_loop)], [\"tek çağrı\", fmt_time(t_vec)]],\n            note=\"Veri artırmada (rastgele döndürme/ölçek) her örneğe ayrı dönüşüm böyle uygulanır.\")\n",
 "x1": "import numpy as np\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.camera()\n\n    def bp():\n        P = need(ns, \"bit_planes\")(img)\n        ensure(P.shape == (8,) + img.shape and set(np.unique(P)) <= {0, 1}, \"(8, H, W) ve yalnız 0/1\")\n        rec = (P.astype(np.int64) * (2 ** np.arange(8))[:, None, None]).sum(0)\n        ensure(np.array_equal(rec, img), \"Σ 2^k · düzlem_k = görüntü\")\n        ensure(np.array_equal(P[7], (img >= 128).astype(np.uint8)), \"düzlem 7 = (img ≥ 128)\")\n    R.test(\"bit_planes: yeniden kurulum\", bp)\n\n    def rq():\n        for b in (1, 3, 5, 8):\n            q = need(ns, \"requantize\")(img, b)\n            ensure(len(np.unique(q)) <= 2 ** b and q.dtype == np.uint8, f\"{b} bit: {len(np.unique(q))} seviye\")\n            ensure(np.all(q <= img) and np.all(img - q < 2 ** (8 - b)), f\"{b} bit: aşağı yuvarlama\")\n        ensure(np.array_equal(ns[\"requantize\"](img, 8), img), \"8 bit özdeşlik\")\n    R.test(\"requantize\", rq)\n",
 "x2": "import numpy as np\nimport cv2\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"cv2/numpy kullanılmıyor\", lambda: no_imports(src, {\"cv2\", \"numpy\", \"scipy\"}))\n    rng = np.random.default_rng(0)\n    B = (rng.random((30, 40)) > 0.03).astype(np.uint8)\n    B[0, :] = 1; B[:, 0] = 1\n    B[5, 7] = 0\n    for metric, dist in ((\"d4\", cv2.DIST_L1), (\"d8\", cv2.DIST_C)):\n        def t(metric=metric, dist=dist):\n            got = np.array(need(ns, \"distance_transform\")(B.tolist(), metric), np.float32)\n            close(got, cv2.distanceTransform(B, dist, 3), atol=0)\n        R.test(f\"{metric} = cv2.distanceTransform\", t)\n",
 "x3": "import numpy as np\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.camera().astype(np.float64)[100:300, 150:330]\n\n    def rot():\n        for deg in (30, -75):\n            got = need(ns, \"rotate_bilinear\")(img, deg)\n            ref = ndimage.rotate(img, deg, reshape=False, order=1, mode=\"constant\", cval=0.0)\n            H, W = img.shape\n            inner = (slice(H // 4, 3 * H // 4), slice(W // 4, 3 * W // 4))\n            close(got[inner], ref[inner], atol=1e-6, what=f\"{deg}° (merkez bölge)\")\n    R.test(\"rotate_bilinear = scipy.ndimage.rotate (saat yönünün tersine)\", rot)\n\n    def aff():\n        rng = np.random.default_rng(0)\n        A = np.array([[0.9, -0.3], [0.25, 1.1]]); t = np.array([5.0, -2.0])\n        src_pts = rng.uniform(0, 100, (12, 2))\n        dst_pts = src_pts @ A.T + t + rng.normal(0, 0.01, (12, 2))\n        Ah, th = need(ns, \"estimate_affine\")(src_pts, dst_pts)\n        close(Ah, A, atol=1e-3); close(th, t, atol=0.05)\n        A3, t3 = ns[\"estimate_affine\"](src_pts[:3], (src_pts[:3] @ A.T + t))\n        close(A3, A, atol=1e-8, what=\"tam 3 nokta → tam çözüm\")\n    R.test(\"estimate_affine (gürültülü 12 nokta, tam 3 nokta)\", aff)\n",
 "x4": "import numpy as np\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü yok\", lambda: no_loops(src))\n    rgb = data.astronaut()[60:316, 140:396].astype(np.float64) / 255\n\n    def t():\n        m = need(ns, \"bayer_mask\")(4, 4)\n        ensure(m.shape == (4, 4, 3) and m[0, 0, 0] and m[0, 1, 1] and m[1, 0, 1] and m[1, 1, 2], \"RGGB yerleşimi\")\n        ensure((m.sum(-1) == 1).all() and m[..., 1].sum() == 8, \"her piksel tek renk, yeşil yarısı\")\n        raw = need(ns, \"mosaic\")(rgb)\n        ensure(raw.shape == rgb.shape[:2], \"ham görüntü tek kanal\")\n        close(raw[0, 0], rgb[0, 0, 0], what=\"(0,0) kırmızı örnek\")\n        close(raw[1, 1], rgb[1, 1, 2], what=\"(1,1) mavi örnek\")\n        out = need(ns, \"demosaic_bilinear\")(raw)\n        mk = ns[\"bayer_mask\"](*raw.shape)\n        close(out[mk], rgb[mk], atol=1e-9, what=\"ölçülen renkler aynen korunur\")\n        P = need(ns, \"psnr\")\n        inner = (slice(4, -4), slice(4, -4))\n        per = [P(out[inner][..., c], rgb[inner][..., c]) for c in range(3)]\n        near = np.repeat(np.repeat(raw[::2, ::2], 2, 0), 2, 1)\n        nn = np.stack([np.repeat(np.repeat(raw[::2, ::2], 2, 0), 2, 1), np.repeat(np.repeat(raw[::2, 1::2], 2, 0), 2, 1), np.repeat(np.repeat(raw[1::2, 1::2], 2, 0), 2, 1)], -1)\n        pn = [P(nn[inner][..., c], rgb[inner][..., c]) for c in range(3)]\n        ensure(per[1] > per[0] and per[1] > per[2], f\"yeşil kanal daha iyi kestirilmeli (2× örnek) {per}\")\n        ensure(min(per) > max(pn) - 1, \"çift doğrusal, en yakın kopyadan iyi\")\n        lum = rgb.mean(-1)\n        g = np.hypot(*np.gradient(lum))\n        edge = g > np.quantile(g, 0.9)\n        chroma_err = lambda im: np.abs((im - im.mean(-1, keepdims=True)) - (rgb - rgb.mean(-1, keepdims=True))).sum(-1)\n        ce = chroma_err(out)\n        R_.table(\"Demozaikleme (astronot kırpıntısı, 256 × 256)\", [\"yöntem\", \"PSNR R\", \"PSNR G\", \"PSNR B\", \"kenarlarda renk hatası\", \"düz bölgelerde renk hatası\"],\n                 [[\"en yakın (2 × 2 kopya)\", *[f\"{v:.1f} dB\" for v in pn], f\"{chroma_err(nn)[edge].mean():.3f}\", f\"{chroma_err(nn)[~edge].mean():.3f}\"],\n                  [\"çift doğrusal\", *[f\"{v:.1f} dB\" for v in per], f\"{ce[edge].mean():.3f}\", f\"{ce[~edge].mean():.3f}\"]],\n                 note=\"Yeşil iki kat örneklendiği için en iyi kestirilen kanal. Renk (kroma) hatası kenarlarda yoğunlaşır: kırmızı ve mavi farklı konumlardan kestirildiği için keskin bir kenarın iki yanı farklı renk kanallarında farklı yerde 'görünür' — yanlış renk saçakları. Daha iyi yöntemler (Malvar–He–Cutler, kenar yönlü enterpolasyon) yeşilin yüksek frekansını kırmızı/maviye aktararak bunu azaltır.\")\n    R_.test(\"Bayer mozaiği ve çift doğrusal demozaikleme\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_1_digital_image as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Kamera sürekli bir parlaklık fonksiyonu $f(x,y)$'yi iki kez ayrıklaştırır:

- **Örnekleme** (uzamsal): bir ızgaradaki noktalarda değer al → $M\times N$ piksel. Çözünürlük düşükse ince ayrıntılar kaybolur ya da **aliasing** olur.
- **Kuantalama** (yoğunluk): her değeri $L=2^k$ seviyeden birine yuvarla. 8 bit (256 seviye) gri görüntülerde standarttır; az bit **yanlış kontur** üretir.

Depolama: $M\cdot N\cdot k$ bit.

**İnterpolasyon** ızgara noktaları arasındaki değeri tahmin eder. $(x, y)$'nin dört komşusu $x_0=\lfloor x\rfloor$, $y_0=\lfloor y\rfloor$, ağırlıklar $a=x-x_0$, $b=y-y_0$:

$$
f(x,y)\approx(1-a)(1-b)f_{00}+a(1-b)f_{01}+(1-a)b\,f_{10}+ab\,f_{11}.
$$

Bikübik interpolasyon her eksende 4 komşuyu Keys çekirdeğiyle ağırlıklar ($a=-0.75$ OpenCV, $-0.5$ birçok başka kütüphane):

$$
W(t)=\begin{cases}(a+2)|t|^3-(a+3)|t|^2+1 & |t|\le1\\ a|t|^3-5a|t|^2+8a|t|-4a & 1<|t|<2\\ 0 & \text{aksi}\end{cases}
$$

**OpenCV'nin piksel merkezi kuralı:** ölçek $s=$ kaynak/hedef boyutu, hedef $x$'in kaynağı $x_s=(x+0.5)s-0.5$, kenarda kırpılır.

**Geometrik dönüşümler** (döndürme, ölçekleme, kaydırma, kesme) bir afin matrisle yazılır. Uygulamada **ters eşleme**: her çıkış pikseli için kaynak koordinatını hesapla ve interpolasyonla oku.

**Komşuluk ve mesafe:** 4-komşular yatay/dikey, 8-komşular çaprazları da içerir; $D_4=|\Delta x|+|\Delta y|$, $D_8=\max(|\Delta x|,|\Delta y|)$, $D_e=\sqrt{\Delta x^2+\Delta y^2}$.

Test görüntüleri: `skimage.data.camera()` (512 × 512) gibi scikit-image ile gelen kamu malı görüntüler; tarayıcıda da yüklenir.

:::sezgi Görüntüyü kim, hangi dalga boyunda üretiyor?
"Görüntü" yalnız görünür ışık değildir; **görüntüleme modalitesi** algılanan fiziksel büyüklüğe göre değişir. Elektromanyetik spektrumda kısa dalgadan uzuna: gama ışını (nükleer tıp, PET; ~10⁻⁶ µm), X-ışını (röntgen, BT; ~10⁻⁴ µm), morötesi (floresan mikroskopi, astronomi), görünür ışık (0.4–0.7 µm; kameralar), kızılötesi (termal kameralar, uydu bitki örtüsü; 1–1000 µm), mikrodalga (sentetik açıklıklı radar, SAR; bulutun içinden görür), radyo (MRI; hidrojen çekirdeklerinin radyo frekansı yanıtı). Elektromanyetik olmayanlar: ultrason (ses yankısı), elektron mikroskobu. Hepsi sonunda aynı şeye dönüşür: bir ızgara üzerinde örneklenmiş, nicemlenmiş sayılar — bu modülün konusu.
:::

:::tanım Tek sensörlü renkli kamera: Bayer deseni
Bir görüntü algılayıcısının her pikseli yalnız ışık **şiddetini** ölçer. Renk için önüne bir **renk süzgeci dizisi** (CFA) konur; en yaygını Bayer deseni (RGGB): 2 × 2'lik her blokta bir kırmızı, iki yeşil, bir mavi süzgeç. Yeşil iki kat örneklenir çünkü insan gözünün parlaklık duyarlılığı en çok yeşildedir (koni hücrelerinin tepkileri M ve L konilerinde örtüşür). Her pikselde eksik iki renk komşulardan **kestirilir** (demozaikleme); kenarlarda yanlış renk (*false color*) ve fermuar (*zipper*) artefaktları bu kestirimden gelir. Ek alıştırma E4'te kendin yapacaksın.
:::

## 1 · Elle

<!-- çözüm -->

Bilineer interpolasyon iki doğrusal interpolasyonun bileşimidir: sıranın önemi yok, önce $y$ sonra $x$ yönünde de aynı sonucu bulursun.

Mesafelerde $D_4\ge D_e\ge D_8$ her zaman geçerli.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Örnekleme ve kuantalama**

- `bytes`: 1024 × 768, 8 bit gri görüntü: bayt
- `levels`: 3 bitlik kuantalamada seviye sayısı
- `q200`: 200 değerini 3 bite indir: $\lfloor 200/32\rfloor\cdot32$

**$I=\begin{bmatrix}10&20\\30&40\end{bmatrix}$ (satır $y$, sütun $x$)**

- `bil`: Bilineer $I(x=0.25,\ y=0.5)$
- `nn`: En yakın komşu $I(x=0.6,\ y=0.4)$
- `src`: OpenCV: 4 piksellik satır 2'ye küçülürken hedef $x=0$'ın kaynak koordinatı $(x+0.5)\cdot s-0.5$

**$p=(0,0)$, $q=(3,4)$**

- `de`: Öklid mesafesi
- `d4`: $D_4$ (şehir bloğu)
- `d8`: $D_8$ (satranç tahtası)

In [ ]:
cevaplar = dict(
    bytes=None,
    levels=None,
    q200=None,
    bil=None,
    nn=None,
    src=None,
    de=None,
    d4=None,
    d8=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"bytes":[786432,0],"levels":[8,0],"q200":[192,0],"bil":[22.5,0],"nn":[20,0],"src":[0.5,0],"de":[5,0],"d4":[7,0],"d8":[4,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; görüntü bir liste listesi (`img[y][x]`), değerler `float`.

- `bilinear_at(img, x, y)` — koordinatlar görüntünün dışına taşarsa kenara kırp (`0 ≤ x ≤ W-1`).
- `resize_nearest(img, H2, W2)` — `cv2.INTER_NEAREST` kuralı: kaynak $\lfloor x\cdot W/W_2\rfloor$.
- `resize_bilinear(img, H2, W2)` — OpenCV'nin piksel merkezi kuralı + `bilinear_at`.

Kontrol, `skimage.data.camera()`'dan kesilen küçük bir parçada `cv2.resize` (float32) ile birebir karşılaştırır.

In [ ]:
%%writefile stage_b.py
def bilinear_at(img, x, y):
    ...


def resize_nearest(img, H2, W2):
    ...


def resize_bilinear(img, H2, W2):
    ...


I = [[10.0, 20.0], [30.0, 40.0]]
print(bilinear_at(I, 0.25, 0.5))          # 22.5
print(resize_bilinear(I, 4, 4)[0])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**.

- `resize_bilinear(img, H2, W2)` — her eksen için kaynak indeksleri ve ağırlıklar vektörü, sonra gelişmiş indeksleme.
- `resize_cubic(img, H2, W2, a=-0.75)` — ayrılabilir: önce satırlarda 4 komşu ağırlıklı toplam, sonra sütunlarda. İndeksler kenarda kırpılır (`BORDER_REPLICATE`).

Kontrol tam boyutlu `camera` görüntüsünde büyütme ve küçültme için `cv2.resize` ile karşılaştırır (tolerans 0.01 gri seviye: tarayıcıdaki WebAssembly OpenCV ağırlıkları biraz farklı yuvarlıyor).

In [ ]:
%%writefile stage_c.py
import numpy as np


def resize_bilinear(img, H2, W2):
    ...


def resize_cubic(img, H2, W2, a=-0.75):
    ...


from skimage import data
img = data.camera().astype(np.float32)
print(resize_bilinear(img, 200, 300).shape, resize_cubic(img, 700, 900).shape)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch ile türevlenebilir geometri.

- `warp_affine(img, A, t)` — `img: (H, W)`; çıkış pikseli $(x, y)$'nin kaynak koordinatı $\begin{bmatrix}x_s\\y_s\end{bmatrix}=A\begin{bmatrix}x\\y\end{bmatrix}+t$ (ters eşleme, piksel biriminde). `F.grid_sample(..., mode="bilinear", padding_mode="zeros", align_corners=False)` için koordinatları normalize et: $x_n=(2x_s+1)/W-1$.
- `register(fixed, moving, steps=300, lr=0.05)` → `(theta, tx, ty)`: `moving` görüntüyü merkez etrafında $\theta$ döndürüp $(t_x, t_y)$ kaydıran parametreleri, `warp_affine(moving, …)` ile `fixed` arasındaki MSE'yi Adam ile küçülterek bul. Kayıp kenarlardan 16 piksel içeride hesaplanmalı: dönmüş görüntünün sıfırla doldurulmuş köşeleri kaybı kirletir ve optimumu birim dönüşüme doğru çeker (dene ve gör). Parametreleri benzer ölçekte tutmak için kaydırmayı `10 × p` olarak parametrize et.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F


def warp_affine(img, A, t):
    """img (H, W), A (2, 2), t (2,): kaynak = A @ [x, y] + t (piksel birimi)."""
    ...


def register(fixed, moving, steps=300, lr=0.05):
    ...


from skimage import data
img = torch.tensor(data.camera(), dtype=torch.float32)[::4, ::4] / 255
print(warp_affine(img, torch.eye(2), torch.tensor([3.0, 0.0])).shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`warp_batch(imgs, A, t)` — `imgs: (B, H, W)`, `A: (B, 2, 2)`, `t: (B, 2)`; her görüntüye kendi dönüşümü, **tek** `grid_sample` çağrısıyla. Döngü yok.

Kontrol, `d`'deki tek görüntülük fonksiyonun döngüyle çağrılmasıyla karşılaştırır ve süreyi ölçer.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def warp_batch(imgs, A, t):
    """imgs (B, H, W), A (B, 2, 2), t (B, 2). Döngü yok."""
    ...


imgs = torch.rand(4, 32, 32)
print(warp_batch(imgs, torch.eye(2).repeat(4, 1, 1), torch.zeros(4, 2)).shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Bit düzlemleri ve yanlış kontur

- `bit_planes(img)` → `(8, H, W)` `uint8`; düzlem $k$, piksellerin $k$. biti ($k=0$ en düşük). Döngü yok (yayınlama ile `>>` ve `& 1`).
- `requantize(img, bits)` — $2^{\text{bits}}$ seviyeye indir: $\lfloor v/2^{8-b}\rfloor\cdot2^{8-b}$.

Gözle: `camera` görüntüsünü 2–3 bite indir; gökyüzündeki bantlar yanlış kontur. En yüksek iki bit düzlemi görüntünün yapısını, alt düzlemler gürültüye benzer bir doku taşır.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def bit_planes(img):
    ...


def requantize(img, bits):
    ...


from skimage import data
img = data.camera()
print(bit_planes(img).shape, np.unique(requantize(img, 2)))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Mesafe dönüşümleri: iki geçişli $D_4$ ve $D_8$

İki geçişli mesafe dönüşümü (saf Python, küçük ikili görüntü): ön planın (1) her pikseli için en yakın arka plan (0) pikseline mesafe.

1. İleri geçiş (sol üstten sağ alta): `d[y][x] = min(d[y][x], d[y-1][x] + 1, d[y][x-1] + 1)` ($D_8$ için çaprazlar `d[y-1][x-1]`, `d[y-1][x+1]` de).
2. Geri geçiş (sağ alttan sol üste) simetrik komşularla.

- `distance_transform(bin_img, metric)` — `metric` `"d4"` ya da `"d8"`; arka plan 0, ön plan başlangıçta büyük bir sayı.

Kontrol `cv2.distanceTransform(…, cv2.DIST_L1, 3)` ve `cv2.DIST_C` ile birebir karşılaştırır.

In [ ]:
%%writefile extra_x2.py
def distance_transform(bin_img, metric):
    ...


B = [[1, 1, 1, 1, 1],
     [1, 1, 1, 1, 1],
     [1, 1, 0, 1, 1],
     [1, 1, 1, 1, 1]]
for row in distance_transform(B, "d4"):
    print(row)

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Döndürme ve bağlantı noktalarından afin kestirimi

- `rotate_bilinear(img, deg)` — görüntü merkezi $c=((W-1)/2,(H-1)/2)$ etrafında saat yönünün tersine döndür; ters eşleme + bilineer; dışarısı 0. NumPy, döngü yok.
- `estimate_affine(src, dst)` — $n\ge3$ eşleşen nokta çiftinden (bağlantı noktaları, *tie points*) $dst\approx A\,src+t$'yi en küçük kareler ile kestir: her nokta iki denklem verir, 6 bilinmeyen. `np.linalg.lstsq`. `(A, t)` döndür.

Görüntü kayıtlamada (registration) noktalar elle ya da öznitelik eşlemeyle (M9.15) bulunur.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def rotate_bilinear(img, deg):
    ...


def estimate_affine(src, dst):
    ...


from skimage import data
img = data.camera().astype(np.float64)
print(rotate_bilinear(img, 30).shape)

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

### E4 · Bayer mozaiği ve demozaikleme: tek sensörden renkli görüntü

Tek sensörlü kameralar her pikselde yalnız bir rengi ölçer (intro'daki Bayer kutusu). NumPy + `scipy.ndimage.convolve`, döngüsüz.

- `bayer_mask(H, W)` → `(H, W, 3)` bool: RGGB (çift satır: R G R G…, tek satır: G B G B…).
- `mosaic(rgb)` → `(H, W)`: her pikselde maskeye düşen kanal.
- `demosaic_bilinear(raw)` → `(H, W, 3)`: her kanalın örneklerini maskeyle seyrek bir görüntüye koy, eksikleri komşu ortalamasıyla doldur. Kırmızı/mavi için $\frac14\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}$, yeşil için $\frac14\begin{bmatrix}0&1&0\\1&4&1\\0&1&0\end{bmatrix}$ (`mode="mirror"`). Bu çekirdekler ölçülen pikselde değeri aynen bırakır (merkez ağırlık × 1, komşular 0), eksik pikselde ise mevcut komşuların ortalamasını verir — nedenini bir 3 × 3 desen üzerinde elle doğrula.
- `psnr(a, b, peak=1.0)`.

In [ ]:
%%writefile extra_x4.py
import numpy as np
from scipy.ndimage import convolve


def bayer_mask(H, W):
    """RGGB: (0,0) R, (0,1) G, (1,0) G, (1,1) B. Dönüş (H, W, 3) bool."""
    ...


def mosaic(rgb):
    ...


def demosaic_bilinear(raw):
    ...


def psnr(a, b, peak=1.0):
    ...

In [ ]:
# Kontrol (E4)
labcheck.run_file("extra_x4.py", CHECKS["x4"])

## Makale ödevi: Çok aşamalı görüntü yeniden örnekleme kütüphanesi

[Keys (1981) — Cubic convolution interpolation; Lanczos yeniden örnekleme](https://doi.org/10.1109/TASSP.1981.1163711)

Kendi `resize(img, size, method)` fonksiyonunu `nearest`, `bilinear`, `bicubic` ve `lanczos4` yöntemleriyle, **küçültmede antialiasing'li** (alan ortalamalı, `cv2.INTER_AREA` eşdeğeri) olarak yaz. Yöntemlerin kalitesini ve aliasing'i karşılaştır.


**Adımlar**

1. Ayrılabilir 1-boyutlu çekirdek fonksiyonları yaz (kutu, üçgen, Keys $a=-0.75$, Lanczos-4) ve her eksende ayrı uygula.
2. Küçültmede çekirdeği ölçek oranı kadar genişlet (antialias); `cv2.INTER_AREA` tam sayı oranlarda alan ortalamasıdır.
3. Bir zone-plate (eş merkezli halka) test görüntüsü üret; 4× küçült ve yöntemlerin aliasing'ini görsel ve sayısal (yüksek frekans enerjisi) karşılaştır.
4. Bir görüntüyü 2× küçültüp tekrar büyüt; her yöntem için orijinale göre PSNR hesapla.

**Kabul ölçütleri**

- `bilinear` ve `bicubic` float32'de `cv2.resize` ile en fazla $10^{-3}$ farklı.
- Tam sayı küçültme oranlarında `area` yöntemi `cv2.INTER_AREA` ile en fazla $10^{-3}$ farklı.
- Zone-plate'te antialias'lı küçültmenin yüksek frekans enerjisi naif bilineerinkinin en az 5 katı düşük.

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, Bölüm 2](https://www.imageprocessingplace.com/) — ©
- [OpenCV — Geometric Image Transformations (resize, warpAffine)](https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html) — Apache-2.0
- [Keys — Cubic convolution interpolation for digital image processing (1981)](https://doi.org/10.1109/TASSP.1981.1163711) — ©
- [Szeliski — Computer Vision: Algorithms and Applications, Bölüm 3.6](https://szeliski.org/Book/) — © (PDF ücretsiz)
- [PyTorch — torch.nn.functional.grid_sample / affine_grid](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.grid_sample.html) — BSD-3-Clause